# imports

In [91]:
import pickle
import os
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import matplotlib as mpl
import hypertools as hyp
import numpy as np
from scipy import optimize
from scipy.signal import resample
from fastdtw import fastdtw
from scipy.spatial.distance import correlation
from scipy.stats import pearsonr
from multiprocessing import Pool

%matplotlib inline
plt.rc('figure', figsize=(12, 8))

### paths

In [77]:
annot_dir = '../../data/annotations_dfs/'
model_dir = '../../data/models/'
audio_dir = '../../data/audio/'
pickles = '../../data/pickles/'

### mapping between subject ID and psiturk ID

In [14]:
with open('../../data/pickles/id_maps.p', 'rb') as f:
    id_maps = pickle.load(f)

In [17]:
# temporary: only run on 5 test subjects
rand5_turkids = {k: id_maps[k] for k in ['MD-013119-A-02', 'MD-102218-A-07', 'MD-020119-A-04',
                                         'MD-102218-B-01', 'MD-020119-B-02']}
id_maps = rand5_turkids

## load annotation dataframes

In [5]:
atlep1_df = pd.read_pickle(annot_dir+'atlep1.p')
atlep2_df = pd.read_pickle(annot_dir+'atlep2.p')
arrdev_df = pd.read_pickle(annot_dir+'arrdev.p')

## load in resampled episode models

In [30]:
atlep1_model = np.load(model_dir+'video/t100_w50/atlep1_model_t100_w50_res.npy')
atlep2_model = np.load(model_dir+'video/t100_w50/atlep2_model_t100_w50_res.npy')
arrdev_model = np.load(model_dir+'video/t100_w50/arrdev_model_t100_w50_res.npy')

## load in manual transcription recall models

In [69]:
with open(pickles+'transcription_comparison_models_w200_resampled.p', 'rb') as f:
    transc_models = pickle.load(f)
    
man_models = {}
for tid, audio_type in transc_models.items():
    man_models[tid] = {}
    for at, transc_type in audio_type.items():
        man_models[tid][at] = transc_type['manual']

In [81]:
# model parameters
n_topics = 100
episode_wsize = 50

# vectorizer parameters
vectorizer_params = {
    'model' : 'CountVectorizer', 
    'params' : {
        'stop_words' : 'english'
    }
}

# topic model parameters
semantic_params = {
    'model' : 'LatentDirichletAllocation', 
    'params' : {
        'n_components' : n_topics,
        'learning_method' : 'batch',
        'random_state' : 0,
    }
}

## functions for modeling and resampling episodes, modeling transcripts

In [7]:
# def find_midpoint_time(df, endframe_time):
#     """
#     returns list of timepoints at middle of each annotation segment
#     """
#     midpoint_times = []
#     for i, tpt in enumerate(df['Onset time']):
#         if i != len(df['Onset time'])-1:
#             midpoint_time = np.mean([tpt, df['Onset time'][i+1]])
#         else:
#             midpoint_time = np.mean([tpt, endframe_time])
#         midpoint_times.append(midpoint_time)

#     return midpoint_times

In [142]:
def transform_recall(transcript, n_topics, recall_wsize, vec_params, sem_params, episode_windows, resample_shape):
    """
    transform recalls based on sliding windows of episode annotations
    """
    # sliding windows of episode annotations and recall words
    recall_w = []

    # split recall transcript into list
    rec_list = transcript.split()
    
    # create overlapping windows of recall_wsize words
    for ix, word in enumerate(rec_list):
        recall_w.append(','.join(rec_list[ix:ix+recall_wsize]))
        
    # create recall model
    recall_model = hyp.tools.format_data(recall_w, vectorizer=vec_params, semantic=sem_params,
                                         corpus=episode_windows)[0]
    
    # return recalled model to corresponding episode model length
    return resample(recall_model, resample_shape)

In [ ]:
def model_resample_episode(episode_df, n_topics, episode_wsize, vec_params, sem_params):
    
    # throw all annotations into bag of words to train model
    episode_bag = episode_df.loc[:,'Narrative details (external events)':'Setting'].apply(lambda x: ', '.join(
        x.fillna('')), axis=1).values.tolist()
    
    # create list for annotation sliding windows (of size w_size)
    episode_w = []
    for idx, sentence in enumerate(episode_bag):
        episode_w.append(','.join(episode_bag[idx:idx+episode_wsize]))
        
    # use hypertools to create episode model
    model = hyp.tools.format_data(episode_w, vectorizer=vec_params, semantic=sem_params, 
                                 corpus=episode_w)[0]
    
    if episode_df == atlep1_df:
        endframe_time = 1466.0
    elif episode_df == atlep2_df:
        endframe_time = 1316.52
    elif episode_df == arrdev_df:
        endframe_time = 1236.6
        
    midpoint_times = find_midpoint_time(episode_df, endframe_time)
    
    new_model = np.empty((int(round(endframe_time)),np.shape(model)[1]))
    
    # loop over topic dimensions
    for dim in range(np.shape(model)[1]):
        # values for given dimension at each timepoint
        single_dim = []
        for tpt in range(np.shape(model)[0]):
            single_dim.append(model[tpt][dim])
        
        # create interpolation function from dimension timeseries
        interp_func = interpolate(midpoint_times, single_dim, fill_value='extrapolate')
        
        # set of new timepoints
        new_tpts = np.arange(int(round(endframe_time)), step=1)
        
        # interpolate single topic dimension trajectory new timescale
        single_dim_res = interp_func(new_tpts)
        
        # fill in array for resampled model
        for ix, new_tpt in enumerate(single_dim_res):
            new_model[ix][dim] = new_tpt
    
    return new_model
    

In [79]:
def get_episode_windows(episode_df, episode_wsize):
    # throw all annotations into bag of words to train model
    episode_bag = episode_df.loc[:,'Narrative details (external events)':'Setting'].apply(lambda x: ', '.join(x.fillna('')), axis=1).values.tolist()

    # create list for annotation sliding windows (of size w_size)
    episode_w = []
    for idx, sentence in enumerate(episode_bag):
        episode_w.append(','.join(episode_bag[idx:idx+episode_wsize]))

    return episode_w

In [82]:
# sliding annotation windows for the three episodes
atlep1_windows = get_episode_windows(atlep1_df, episode_wsize)
atlep2_windows = get_episode_windows(atlep2_df, episode_wsize)
arrdev_windows = get_episode_windows(arrdev_df, episode_wsize)

In [143]:
def compare_models(model1, model2):
    return pearsonr(pd.DataFrame(model1).T.corr().values.ravel(), pd.DataFrame(model2).T.corr().values.ravel())[0]

## search over auto recall w_sizes to find max corr to manual recall models with existing wsize (200 words)

In [144]:
recall_wsizes = [100, 125, 150, 175, 200, 225]

In [ ]:
corrs = []
for w_size in recall_wsizes:
    wsize_corrs = []
    for tid, audio_type in man_models.items():
        for a_t, man_model in audio_type.items():
            if a_t != 'prediction':
            
                # assign correct filepath
                if os.path.isdir(os.path.join(audio_dir,'room1',tid)):
                    filepath = os.path.join(audio_dir,'room1',tid,tid+'-'+a_t+'.wav.txt')
                else:
                    filepath = os.path.join(audio_dir,'room2',tid,tid+'-'+a_t+'.wav.txt')

                # assign correct sliding window corpus
                for sid, data in id_maps.items():
                    for ses, turkid in data.items():
                        if turkid == tid:
                            if ses == 'session 1' or a_t == 'delayed':
                                ep_winds = atlep1_windows
                            elif a_t == 'recall' and 'A' in sid:
                                ep_winds = atlep2_windows
                            else:
                                ep_winds = arrdev_windows
                                
                with open(filepath, 'r') as f:
                    transcript = f.read()

                # transform auto transcription
                auto_model = transform_recall(transcript, 100, w_size, vectorizer_params, semantic_params, ep_winds,
                                np.shape(man_model)[0])

                wsize_corrs.append(compare_models(man_model, auto_model))
                
    avg_wsize_corr = np.mean(wsize_corrs)
    print(avg_wsize_corr)
    corrs.append(avg_wsize_corr)

0.5410937246787313
0.5978056504722101
